In [3]:
import joblib
import numpy as np
from model.create_dataset import vectorize_state
from model.game import sample_until, sample_every, sample
import os
import json
from scipy import stats
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import random

In [4]:
with open('item.json') as f:
    item_data = json.load(f)

In [5]:
model = joblib.load('model.pkl')

In [26]:
root = 'dataset/test'

all_files = []

for dirpath, dirnames, filenames in os.walk(root):
    for filename in filenames:
        all_files.append(os.path.join(dirpath, filename))

In [7]:
def format_wpa_summary(mean, me):
    def format_percentage(number):
        return f'{round(number * 100, 2)}%'

    return f'{format_percentage(mean)} ± {format_percentage(me)}'

In [8]:
def calc_first_event_wpa(callback):
    winrates = []
    wpes = []
    
    # Loop through all games
    for game_path in all_files:
        with open(game_path) as f:
            game = json.load(f)
        state, outcome = sample_until(game, callback)
        # outcome is a boolean that depics which team got the event, False for red team True for blue team
        if state:
            winrates.append(state['win'] == outcome)
            v = vectorize_state(state)
            v = np.array(v, dtype=np.float32)
            v = np.expand_dims(v, 0)
            prob = model.predict_proba(v)[0, outcome].item()
            wpes.append(prob)

    return np.array(winrates), np.array(wpes)

In [9]:
def calc_average_event_wpa(callback):
    winrates = []
    wpes = []
    
    # Loop through all games
    for game_path in all_files:
        with open(game_path) as f:
            game = json.load(f)
        results = sample_every(game, callback)
        # outcome is a boolean that depics which team got the event, False for red team True for blue team
        for state, outcome in results:
            if state:
                winrates.append(state['win'] == outcome)
                v = vectorize_state(state)
                v = np.array(v, dtype=np.float32)
                v = np.expand_dims(v, 0)
                prob = model.predict_proba(v)[0, outcome].item()
                wpes.append(prob)

    return np.array(winrates), np.array(wpes)

In [29]:
def calculate_wpa_ci(wr, we):
    d = wr - we
    n = len(d)
    mean = d.mean()
    std = d.std(ddof=1)
    sem = std / np.sqrt(n)
    t_value = stats.t.ppf(0.975, df=n-1)
    me = t_value * sem + 0.0042
    return mean, me

In [11]:
def get_item_callback(item_id):
    def item_callback(state, event):
        if event['type'] == 'ITEM_PURCHASED' and event['itemId'] == item_id:
            return True, 1 - int((event['participantId'] - 1) / 5)
        return False, None

    return item_callback

In [12]:
def champion_kill(state, event):
    if event['type'] == 'CHAMPION_KILL':
        return True, 1 - int((event['killerId'] - 1) / 5)
    return False, None

In [13]:
def baron(state, event):
    if event['type'] == 'ELITE_MONSTER_KILL' and event['monsterType'] == 'BARON_NASHOR':
        return True, 1 - int((event['killerId'] - 1) / 5)
    return False, None

In [14]:
def elder(state, event):
    if event['type'] == 'ELITE_MONSTER_KILL' and event['monsterType'] == 'DRAGON' and event['monsterSubType'] == 'ELDER_DRAGON':
        return True, 1 - int((event['killerId'] - 1) / 5)
    return False, None

In [15]:
def dragon(state, event):
    if event['type'] == 'ELITE_MONSTER_KILL' and event['monsterType'] == 'DRAGON' and event['monsterSubType'] != 'ELDER_DRAGON':
        return True, 1 - int((event['killerId'] - 1) / 5)
    return False, None

In [16]:
def void_grubs(state, event):
    if event['type'] == 'ELITE_MONSTER_KILL' and event['monsterType'] == 'HORDE':
        return True, 1 - int((event['killerId'] - 1) / 5)
    return False, None

In [17]:
def riftherald(state, event):
    if event['type'] == 'ELITE_MONSTER_KILL' and event['monsterType'] == 'RIFTHERALD':
        return True, 1 - int((event['killerId'] - 1) / 5)
    return False, None

In [18]:
def top_tower(state, event):
    if event['type'] == 'BUILDING_KILL' and event['buildingType'] == 'TOWER_BUILDING' and event['laneType'] == 'TOP_LANE' and event['killerId'] != 0 and sum(state['teams'][0]['towers']) == 9 and sum(state['teams'][1]['towers']) == 9:
        return True, 1 - int((event['killerId'] - 1) / 5)
    return False, None

In [19]:
def mid_tower(state, event):
    if event['type'] == 'BUILDING_KILL' and event['buildingType'] == 'TOWER_BUILDING' and event['laneType'] == 'MID_LANE' and event['killerId'] != 0 and sum(state['teams'][0]['towers']) == 9 and sum(state['teams'][1]['towers']) == 9:
        return True, 1 - int((event['killerId'] - 1) / 5)
    return False, None

In [20]:
def bot_tower(state, event):
    if event['type'] == 'BUILDING_KILL' and event['buildingType'] == 'TOWER_BUILDING' and event['laneType'] == 'BOT_LANE' and event['killerId'] != 0 and sum(state['teams'][0]['towers']) == 9 and sum(state['teams'][1]['towers']) == 9:
        return True, 1 - int((event['killerId'] - 1) / 5)
    return False, None

In [30]:
item_ids = [3363, 3157, 3916, 3165]

for item_id in item_ids:
    wr, we = calc_average_event_wpa(get_item_callback(item_id))
    mean, me = calculate_wpa_ci(wr, we)
    item_name = item_data['data'][str(item_id)]['name']
    print(f'{item_name}: ' + format_wpa_summary(mean, me))

Farsight Alteration: 0.39% ± 0.61%
Zhonya's Hourglass: -1.33% ± 0.7%
Oblivion Orb: 0.36% ± 0.92%
Morellonomicon: -1.52% ± 1.37%


In [31]:
item_ids = [2031, 2055, 1082, 3364]

for item_id in item_ids:
    wr, we = calc_average_event_wpa(get_item_callback(item_id))
    mean, me = calculate_wpa_ci(wr, we)
    item_name = item_data['data'][str(item_id)]['name']
    print(f'{item_name}: ' + format_wpa_summary(mean, me))

Refillable Potion: 0.24% ± 0.65%
Control Ward: -0.1% ± 0.49%
Dark Seal: 1.64% ± 0.73%
Oracle Lens: 0.15% ± 0.58%


In [32]:
wr, we = calc_average_event_wpa(void_grubs)
mean, me = calculate_wpa_ci(wr, we)
print(format_wpa_summary(mean, me))

1.65% ± 0.6%


In [33]:
wr, we = calc_first_event_wpa(riftherald)
mean, me = calculate_wpa_ci(wr, we)
print(format_wpa_summary(mean, me))

-0.46% ± 0.71%


In [34]:
wr, we = calc_average_event_wpa(baron)
mean, me = calculate_wpa_ci(wr, we)
print(format_wpa_summary(mean, me))

4.89% ± 0.65%


In [35]:
wr, we = calc_average_event_wpa(elder)
mean, me = calculate_wpa_ci(wr, we)
print(format_wpa_summary(mean, me))

9.39% ± 1.25%


In [36]:
wr, we = calc_average_event_wpa(dragon)
mean, me = calculate_wpa_ci(wr, we)
print(format_wpa_summary(mean, me))

4.15% ± 0.56%


In [37]:
wr, we = calc_first_event_wpa(champion_kill)
mean, me = calculate_wpa_ci(wr, we)
print(format_wpa_summary(mean, me))

7.1% ± 0.76%


In [38]:
wr, we = calc_first_event_wpa(top_tower)
mean, me = calculate_wpa_ci(wr, we)
print(format_wpa_summary(mean, me))

3.22% ± 0.87%


In [39]:
wr, we = calc_first_event_wpa(mid_tower)
mean, me = calculate_wpa_ci(wr, we)
print(format_wpa_summary(mean, me))

5.6% ± 1.09%


In [40]:
wr, we = calc_first_event_wpa(bot_tower)
mean, me = calculate_wpa_ci(wr, we)
print(format_wpa_summary(mean, me))

5.96% ± 0.89%


In [41]:
item_ids = [3047, 3111, 3158, 3006, 3009, 3020]

for item_id in item_ids:
    wr, we = calc_average_event_wpa(get_item_callback(item_id))
    mean, me = calculate_wpa_ci(wr, we)
    item_name = item_data['data'][str(item_id)]['name']
    print(f'{item_name}: ' + format_wpa_summary(mean, me))

Plated Steelcaps: 0.6% ± 0.64%
Mercury's Treads: 0.81% ± 0.65%
Ionian Boots of Lucidity: 0.3% ± 0.64%
Berserker's Greaves: 0.28% ± 0.66%
Boots of Swiftness: 0.5% ± 0.69%
Sorcerer's Shoes: 0.19% ± 0.64%


In [42]:
wr, we = calc_first_event_wpa(baron)
mean, me = calculate_wpa_ci(wr, we)
print(format_wpa_summary(mean, me))

4.11% ± 0.68%


In [43]:
wr, we = calc_average_event_wpa(champion_kill)
mean, me = calculate_wpa_ci(wr, we)
print(format_wpa_summary(mean, me))

4.84% ± 0.46%
